In [ ]:
import pygmt
import numpy as np
import pandas as pd
import meshio
import utils_plot as utp

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_locking/"

# # Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
# obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# # the processed data has the unit of m/yr that was converted from mm/yr
# df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
#                    names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
#                           'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])

filename = "data/Kyriakopoulos_novolcano.csv"
gps_data = pd.read_csv(datadir + filename, sep=",")

# Display the column names to verify
print(gps_data.columns)

# Display the first few rows
print(gps_data.head())

# Access the data you need for plotting
stations = gps_data['Site'].values
lons = gps_data['Lon'].values
lats = gps_data['Lat'].values

# Example: Extract as numpy array for PyGMT plotting
gps_stations = np.column_stack([lons, lats, stations])
# print(gps_stations.head())


In [ ]:
# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
print(trench.head())

In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
print(volcano.head())


In [ ]:
def read_plate_boundaries(filename):
    """
    Read the PB2002 plate boundary file and return a dictionary of boundary segments.
    
    Returns:
        dict: Dictionary with boundary names as keys and lists of (lon, lat) arrays as values
    """
    boundaries = {}
    current_boundary = None
    current_segment = []
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
            
            # Check for segment separator
            if line.startswith('***'):
                # Save current segment if it exists
                if current_boundary and current_segment:
                    if current_boundary not in boundaries:
                        boundaries[current_boundary] = []
                    boundaries[current_boundary].append(np.array(current_segment))
                    current_segment = []
                continue
            
            # Check for header line (boundary name)
            if not line[0].isdigit() and not line[0] in ['+', '-', ' ']:
                # Extract boundary name (first part before spaces)
                current_boundary = line.split()[0]
                continue
            
            # Parse coordinate line
            try:
                parts = line.split(',')
                if len(parts) == 2:
                    lon = float(parts[0])
                    lat = float(parts[1])
                    current_segment.append([lon, lat])
            except ValueError:
                continue
    
    # Don't forget the last segment
    if current_boundary and current_segment:
        if current_boundary not in boundaries:
            boundaries[current_boundary] = []
        boundaries[current_boundary].append(np.array(current_segment))
    
    return boundaries

# Read the plate boundaries
boundaries = read_plate_boundaries('/home/staff/chao/SSEinv/PlateBoundary/PB2002_boundaries.dig_twb.txt')

# # Print available boundaries
# print("Available plate boundaries:")
# for boundary_name in sorted(boundaries.keys()):
#     num_segments = len(boundaries[boundary_name])
#     print(f"  {boundary_name}: {num_segments} segments")


In [ ]:

# Create a global map showing all plate boundaries
fig = pygmt.Figure()

# Plot global map
fig.basemap(
    region="g",  # Global
    projection="N180/20c",  # Robinson projection, 20 cm wide
    frame=["af", "WSnE"],
)

fig.coast(
    land="lightgray",
    water="lightblue",
    shorelines="0.25p,black",
)

# Plot all plate boundaries
# Different colors for different boundary types
boundary_styles = {
    'default': '1p,red',  # Default style
}

for boundary_name, segments in boundaries.items():
    for segment in segments:
        if len(segment) > 0:
            lons = segment[:, 0]
            lats = segment[:, 1]
            
            fig.plot(
                x=lons,
                y=lats,
                pen="0.8p,red",
            )

fig.show()
# fig.savefig("global_plate_boundaries.png", dpi=300)

In [ ]:
fig = pygmt.Figure()

# First command: pscoast
fig.coast(
    region="g",
    projection="N180/22c",
    resolution="l",
    area_thresh=5000,
    land="200",
)

# Second command: psxy
fig.plot(
    data="/home/staff/chao/SSEinv/PlateBoundary/pb2002_boundaries.gmt",
    pen="0.5p,red",
    style="f0.25c/3p",
    fill="red",
    frame=['a30', '+t"Bird\'s 2003 Plate Boundaries"'],
)

fig.show()
# fig.savefig("Bird_PlateBoundaries.png", dpi=300)

In [ ]:
def read_gmt_segments(filename, header_filter=None):
    """
    Read GMT file and filter segments by header.
    
    Parameters:
    -----------
    filename : str
        Path to GMT file
    header_filter : str or None
        String to match in segment header (e.g., "PM-CA")
        If None, returns all segments
    
    Returns:
    --------
    list of dict: Each dict contains 'header', 'style', 'coords'
    """
    segments = []
    current_segment = None
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            
            # Skip empty lines and comments
            if not line or line.startswith('#'):
                continue
            
            # Segment header
            if line.startswith('>'):
                # Check if it's a style line (starts with > -)
                if line.startswith('> -'):
                    # This is a style specification for current segment
                    if current_segment is not None:
                        # Extract style (e.g., "-Sf0.45/3p" -> "f0.45/3p")
                        style_part = line[3:].strip()  # Remove "> -"
                        current_segment['style'] = style_part
                    continue
                
                # Save previous segment if it matches filter
                if current_segment is not None and current_segment['coords']:
                    if header_filter is None or header_filter in current_segment['header']:
                        segments.append(current_segment)
                
                # Start new segment (boundary name line)
                current_segment = {
                    'header': line,
                    'style': None,
                    'coords': []
                }
                
            else:
                # Coordinate line
                if current_segment is not None:
                    try:
                        parts = line.split()
                        if len(parts) >= 2:
                            lon = float(parts[0])
                            lat = float(parts[1])
                            current_segment['coords'].append([lon, lat])
                    except ValueError:
                        continue
        
        # Don't forget last segment
        if current_segment is not None and current_segment['coords']:
            if header_filter is None or header_filter in current_segment['header']:
                segments.append(current_segment)
    
    return segments

In [ ]:
# Define the region for the main map (from your image)
main_region = [-87, -84, 8.5, 11.5]
main_size = 15  # cm
main_projection = f"M{main_size}c"  # Mercator projection, 15 cm width

# Create the figure
fig = pygmt.Figure()

pygmt.config(
    # MAP_FRAME_TYPE="plain", 
    # FONT_TITLE="10p,Helvetica-Bold,black", 
    # MAP_TITLE_OFFSET="-0.2c",
    # MAP_SCALE_HEIGHT="3p", 
    # FONT_ANNOT_PRIMARY="6p", 
    # FONT_ANNOT_SECONDARY="10p",
    FORMAT_GEO_MAP="ddd.x",
    )

# ============================================================================
# MAIN PANEL: Detailed map with bathymetry
# ============================================================================

# Plot high-resolution topography/bathymetry
fig.grdimage(
    grid="@earth_relief_15s",  # 15 arc-second resolution
    region=main_region,
    projection=main_projection,
    # cmap="globe",  # legacy world-map palettes
    # cmap="relief",  # Color palette for topography/bathymetry
    # cmap="topo",  # Color palette for topography/bathymetry
    # cmap="geo",  # General topo+bathy
    # cmap="sealand",  # General topo+bathy
    # cmap="batlow",  # Publication-grade
    # cmap="/home/staff/chao/SSEinv/Nicoya/plateinterface/ETOPO1.cpt",  # Color palette for topography/bathymetry
    # cmap="/home/staff/chao/SSEinv/Nicoya/plateinterface/wiki-2.0.cpt",  # Color palette for topography/bathymetry
    cmap="/home/staff/chao/SSEinv/Nicoya/plateinterface/afrikakarte.cpt",
    shading=True,  # Add hillshading
    frame=["WSne", "xa0.5f0.25", "ya0.5f0.25"],  # Frame with tick marks
)

# Add scale bar (AFTER main map elements)
scale_lon, scale_lat = main_region[0]+0.3, main_region[-1]-0.2
mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + \
    "+w50k+f+l"  # 50 km scale bar

# Add coastlines
fig.coast(
    region=main_region,
    projection=main_projection,
    shorelines="0.25p,black",
    # rivers="a/0.25p,blue",  # Major rivers (a=all permanent rivers)
    # lakes="lightblue",  # Fill lakes
    lakes="cyan",  # Fill lakes
    borders="1/0.15p,black",  # Country borders
    land=None,  # Don't fill land (already have topography)
    water=None,  # Don't fill water
    map_scale=mapscale_str,
)

# ============================================================================
# Add tectonic features
# ============================================================================
# Plot MAT trench line
fig.plot(x=trench['lon'], y=trench['lat'], pen="2p,black", style="f0.5i/0.1i+l+t", 
    fill="black",)

# Add "Middle American Trench" text with tilted angle
fig.text(x=-86.4, y=10.0, text="Middle American Trench", font="12p,Helvetica-Bold,black",
    angle=-45,  # Rotation angle in degrees (counterclockwise)
    justify="LB", fill="white@20",)

# Read only PM-CA segments
pm_ca_segments = read_gmt_segments(
    "/home/staff/chao/SSEinv/PlateBoundary/pb2002_boundaries.gmt",
    header_filter="PM-CA")
# print(f"Found {len(pm_ca_segments)} PM-CA segments")
# for segment in pm_ca_segments:
segment = pm_ca_segments[1]  # Just plot the first segment for demonstration
coords = np.array(segment['coords'])
style_str = segment['style'][1:]
# print(f"Plotting segment with style: {style_str}")
# Plot the segment
fig.plot(x=coords[:, 0], y=coords[:, 1], pen="1.5p,black", fill="black",
    # style=style_str,
    style="f0.5i/0.1i",
)

# plot local slab interface contours from CK
grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=main_region, 
    spacing=(0.05, 0.05),)
fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="10+f7p+gwhite@20", pen="0.5p,black",
    # label_placement="d6c",
    label_placement="lLB/CT",) 

# Plot volcanoes
fig.plot(x=volcano['lon'], y=volcano['lat'], style="kvolcano/0.5c", fill="red", pen="0.3p,black")

# Plot GPS stations
fig.plot(x=gps_data['Lon'].iloc[:26]*(-1.0), y=gps_data['Lat'].iloc[:26], fill="dodgerblue", pen="0.5p,black",
    # style="s0.3c",  # square
    style="D0.25c",  # diamond    
)
fig.plot(x=gps_data['Lon'].iloc[27:]*(-1.0), y=gps_data['Lat'].iloc[27:], fill="dodgerblue", pen="0.5p,black",
    # style="s0.3c",  # square,
    style="c0.25c",  # circle,
)

# # Add GPS station labels
# for idx, row in gps_data.iterrows():
#     fig.text(x=row['Lon']*(-1.0), y=row['Lat'], text=row['Site'], font="5p,Helvetica,black",
#         fill="white@30", offset="0c/0.2c",)

# Add SSE contour from literature
sse_data = pd.read_csv('/home/staff/chao/SSEinv/Nicoya/data/Outerbridge_etal2010JGR_SSE_range.csv', 
                       header=None, names=['Lon', 'Lat'])
fig.plot(x=sse_data['Lon'], y=sse_data['Lat'], pen="1.5p,yellow",
    # fill="red@70",  # Semi-transparent red fill
    close=True,  # Close the polygon
)

# Add locking contours from Kyriakopoulos & Newman (2016)
lock05_data = pd.read_csv('/home/staff/chao/SSEinv/Nicoya/data/Kyriakopoulos&Newman2016JGRSE_0.5locking.csv', 
                       header=None, names=['Lon', 'Lat'])
fig.plot(x=lock05_data['Lon'], y=lock05_data['Lat'], pen="1.5p,gray40",
    # fill="red@70",  # Semi-transparent red fill
    close=True,  # Close the polygon
)
lock09_data = pd.read_csv('/home/staff/chao/SSEinv/Nicoya/data/Kyriakopoulos&Newman2016JGRSE_0.9locking.csv', 
                       header=None, names=['Lon', 'Lat'])
fig.plot(x=lock09_data['Lon'], y=lock09_data['Lat'], pen="1.5p,black",
    # fill="red@70",  # Semi-transparent red fill
    close=True,  # Close the polygon
)

# Add coseismic slip contours from Kyriakopoulos & Newman (2016)
coseis2012_data = pd.read_csv('/home/staff/chao/SSEinv/Nicoya/data/Kyriakopoulos&Newman2016JGRSE_1.5mcoseis.csv',
                              header=None, names=['Lon', 'Lat'])
fig.plot(x=coseis2012_data['Lon'], y=coseis2012_data['Lat'], pen="1.5p,red",
    fill="red@60",  # Semi-transparent red fill
    close=True,  # Close the polygon
)

mt_source = "gCMT"
# mt_source = "Wphase"

# Actual event location
if mt_source == "gCMT":
    # https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2012&mo=9&day=1&otype=ymd&oyr=2012&omo=9&oday=10&jyr=1976&jday=1&ojyr=1976&ojday=1&nday=1&lmw=7&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0
    actual_lon = -85.64   # from GCMT catalog, centroids!
    actual_lat = 10.00
    strike, dip, rake = 308, 17, 102
    magnitude = 7.6
elif mt_source == "Wphase":
    # https://earthquake.usgs.gov/earthquakes/eventpage/usp000jrsw/moment-tensor
    actual_lon = -85.315    # from USGS origin
    actual_lat = 10.085 
    strike, dip, rake = 299, 18, 86
    magnitude = 7.58

# Offset beach ball location
offset_lon = -85.25
offset_lat = 9.4

# Draw connecting line first
fig.plot(x=[actual_lon, offset_lon], y=[actual_lat, offset_lat], pen="1p,black",)

# Mark actual event location (optional small circle)
fig.plot(x=[actual_lon], y=[actual_lat], style="a0.5c", fill="red", pen="0.5p,black",)

# Add focal mechanism, USGS W-phase solution
fig.meca(
    spec=dict(longitude=offset_lon, latitude=offset_lat, depth=30.0,
        strike=strike, dip=dip, rake=rake, magnitude=magnitude),
    scale="0.5c", convention="aki", compressionfill="red", extensionfill="white", pen="0.5p,black",)
fig.text(x=offset_lon, y=offset_lat+0.12, text="2012 Mw=7.6", font="9p,Helvetica,black",
    justify="CM",fill="white@20",)

# ============================================================================
# Add motion arrow (N20°E direction)
# ============================================================================

# Define arrow starting position
arrow_start_lon = -86.0
arrow_start_lat = 8.75

# Calculate arrow end point for N20°E direction
# N20°E means 20° east of north, or azimuth = 20°
# In standard math coordinates: angle from east = 90° - 20° = 70°

arrow_length = 1  # Length in degrees (adjust as needed)
azimuth = 20  # N20°E
azimuth = 25  # N25°E, The convergence direction is consistently ~N22°–25°E across all modern models

# Create vector data: x, y, direction, length
# Direction in degrees (0=east, 90=north, etc.)
vector_data = pd.DataFrame({
    'lon': [arrow_start_lon],
    'lat': [arrow_start_lat],
    'direction': [90 - azimuth],  # Convert azimuth to math angle
    'length': [arrow_length]
})
vector_data_dup = pd.DataFrame({
    'lon': [arrow_start_lon+0.75],
    'lat': [arrow_start_lat-0.3],
    'direction': [90 - azimuth],  # Convert azimuth to math angle
    'length': [arrow_length]
})

fig.plot(
    data=vector_data,
    style="v0.8c+ea+a45+gwhite+h0",  # Vector: 0.4cm head, end arrow, 40° angle
    pen="10p,white", fill="white", 
    incols="0,1,2,3",  # lon, lat, direction, length
)
# fig.plot(data=vector_data_dup, style="v0.8c+ea+a45+gwhite+h0", pen="10p,white", fill="white",
#     incols="0,1,2,3", no_clip=True,)

# Add text
fig.text(x=arrow_start_lon+0.015, y=arrow_start_lat+0.05, text="83±2 mm/yr", font="9p,Helvetica,black",
    angle=90 - azimuth, justify="LM",)

fig.text(x=-84.3, y=10.6, text="Costa", font="16p,Helvetica-BoldOblique,black",
    angle=-45, justify="LB",fill="white@20",)
fig.text(x=-84.4, y=10.5, text="Rica", font="16p,Helvetica-BoldOblique,black",
    angle=-45, justify="LB",fill="white@20",)

fig.text(x=-86.9, y=9.9, text="Cocos", font="18p,Helvetica-Bold,white", justify="LM",fill="black@70",)
fig.text(x=-86.9, y=9.75, text="Plate", font="18p,Helvetica-Bold,white", justify="LM",fill="black@70",)
fig.text(x=-85.75, y=11.35, text="Caribbean", font="16p,Helvetica-Bold,white", justify="LM",fill="black@70",)
fig.text(x=-85.75, y=11.2, text="Plate", font="16p,Helvetica-Bold,white", justify="LM",fill="black@70",)
fig.text(x=-84.6, y=9.15, text="Panama", font="16p,Helvetica-Bold,white", justify="LM", fill="black@70",)
fig.text(x=-84.6, y=9.0, text="Plate", font="16p,Helvetica-Bold,white", justify="LM", fill="black@70",)

# Manual legend - more reliable
legend_x = main_region[0] + 0.05
legend_y = main_region[-2] + 0.05
box_width = 0.85
box_height = 0.9

# Draw background box
fig.plot(
    x=[legend_x, legend_x + box_width, legend_x + box_width, legend_x, legend_x],
    y=[legend_y, legend_y, legend_y + box_height, legend_y + box_height, legend_y],
    fill="white@20", pen="0.25p,black",)

# Volcano symbol
fig.plot(x=[legend_x + 0.1], y=[legend_y + 0.1], fill="red", style="kvolcano/0.5c", pen="0.3p,black",)
fig.text(x=legend_x + 0.2, y=legend_y + 0.1, text="Holocene Volcano", font="9p,Helvetica,black",
         justify="LM",)

# GPS Station symbol
fig.plot(x=[legend_x + 0.1], y=[legend_y + 0.2], style="c0.25c", fill="dodgerblue", pen="0.5p,black")
fig.text(x=legend_x + 0.2, y=legend_y + 0.2, text="Continuous GNSS", font="9p,Helvetica,black",
         justify="LM",)

fig.plot(x=[legend_x + 0.1], y=[legend_y + 0.3], style="D0.25c", fill="dodgerblue", pen="0.5p,black")
fig.text(x=legend_x + 0.2, y=legend_y + 0.3, text="Campaign GNSS", font="9p,Helvetica,black",
         justify="LM",)

fig.plot(x=[legend_x+0.05, legend_x+0.15], y=[legend_y+0.4, legend_y+0.4], pen="1.5p,black")
fig.text(x=legend_x+0.2, y=legend_y+0.4, text="90% locking area", font="9p,Helvetica,black",
         justify="LM")

fig.plot(x=[legend_x+0.05, legend_x+0.15], y=[legend_y+0.5, legend_y+0.5], pen="1.5p,gray40")
fig.text(x=legend_x+0.2, y=legend_y+0.5, text="50% locking area", font="9p,Helvetica,black", 
         justify="LM")

fig.plot(x=[legend_x+0.05, legend_x+0.15], y=[legend_y+0.6, legend_y+0.6], pen="1.5p,yellow")
fig.text(x=legend_x+0.2, y=legend_y+0.6, text="2010 SSE area", font="9p,Helvetica,black", 
         justify="LM")

fig.plot(x=[legend_x+0.05, legend_x+0.15], y=[legend_y+0.7, legend_y+0.7], pen="1.5p,red")
fig.text(x=legend_x+0.2, y=legend_y+0.7, text="2012 coseismic area", font="9p,Helvetica,black", 
         justify="LM")

fig.plot(x=legend_x+0.1, y=legend_y+0.8, style="a0.4c", fill="red", pen="0.5p,black",)
if mt_source == "gCMT":
    fig.text(x=legend_x+0.2, y=legend_y+0.8, text="2012 Mw7.6 centroid", font="9p,Helvetica,black", 
            justify="LM")
elif mt_source == "Wphase":
    fig.text(x=legend_x+0.2, y=legend_y+0.8, text="2012 Mw7.6 epicenter", font="9p,Helvetica,black", 
            justify="LM")

# ============================================================================
# INSET MAP: Regional context with Mercator projection
# ============================================================================

# # First, draw the circular background using paper coordinates BEFORE shifting
# # The inset will be at position (10.5c, 9c) relative to current origin
# # So draw circle there first
# # with fig.set_panel(panel=0):  # Ensure we're in the right context
# fig.plot(
#     data=[[12.8, 12.8]],  # Position where inset will be centered (in paper cm)
#     region=[0, 20, 0, 20],  # Paper space
#     projection="x1c",  # Linear projection, 1:1
#     style="c4.5c",  # Circle, 2.1 cm radius
#     fill="white@20",
#     # pen="0.5p,gray50",
# )

# # Use orthographic projection centered on Central America
# center_lon = -85.5
# center_lat = 10
# inset_size = "4c"

# # Draw the orthographic projection on top
# fig.basemap(
#     region="g",  # Global
#     projection=f"G{center_lon}/{center_lat}/{inset_size}+z60",  # Orthographic, 4 cm diameter
#     frame=["g30"],  # Grid every 30°, annotate every 30°, frame on all sides
# )

# Draw a rectangular background BEFORE shifting
# The inset will be at position (10.5c, 9c) with size 4cm x ~4cm
# Calculate approximate height based on lat range (0 to 25) / lon range (-100 to -70) ratio

inset_region = [-95, -80, 5, 15]  # Central America region
# inset_region = [-110, -65, -5, 30]  # Include Mexico to Colombia

inset_size = 4  # cm
inset_height = 4 * (inset_region[-1] - inset_region[-2]) / (inset_region[1] - inset_region[0])  # Approximately 3.3 cm

inset_center_x = 12.3
inset_center_y = 13.2
rect_width = 5.2
rect_height = rect_width*(inset_height/inset_size)

# Rectangle corners
x_corners = np.array([-1, 1, 1, -1, -1]) * rect_width / 2 + inset_center_x
y_corners = np.array([-1, -1, 1, 1, -1]) * rect_height / 2 + inset_center_y

fig.plot(x=x_corners, y=y_corners, region=[0, 20, 0, 20], projection="x1c",
    fill="white@20",
    # pen="0.5p,gray50",
)

# Shift to create inset in top-right corner
fig.shift_origin(xshift="10.5c", yshift="12c")

inset_projection = f"M{inset_size}c"

# Set thicker frame for inset
with pygmt.config(MAP_FRAME_TYPE="plain", 
                  MAP_FRAME_PEN="0.5p,black", 
                  MAP_GRID_PEN="0.5p,dimgray,--",
                  FONT_ANNOT_PRIMARY="7p", 
                #   FONT_ANNOT_SECONDARY="10p",
                  FORMAT_GEO_MAP="ddd"
                  ):  # Thicker frame (1.5 points)
    # Draw the inset map
    fig.basemap(
        region=inset_region,
        projection=inset_projection,  # Mercator, 4 cm width
        frame=["WSne", "a5f5g5"],  # Frame with grid every 10 degrees
    )

    # Add land in the inset
    fig.coast(
        region=inset_region,
        projection=inset_projection,  # Mercator, 4 cm width
        # frame=["WSne", "a5f5g5"],  # Frame with grid every 10 degrees
        land="darkgray",
        # water="white",
        water="lightblue",
        # shorelines="0.25p,black",
        # borders="1/0.25p,black",
        area_thresh=10000,
    )

    # # Plot plate boundaries in your region
    # for boundary_name, segments in boundaries.items():
    #     for segment in segments:
    #         if len(segment) > 0:
    #             lons = segment[:, 0]
    #             lats = segment[:, 1]
                
    #             # Only plot if segment intersects with region
    #             if (lons.min() < inset_region[1] and lons.max() > inset_region[0] and
    #                 lats.min() < inset_region[3] and lats.max() > inset_region[2]):

    #                 fig.plot(
    #                     x=lons,
    #                     y=lats,
    #                     region=inset_region,
    #                     projection=f"M{inset_size}",  # Mercator, 4 cm width
    #                     pen="1.5p,black",  # Thicker, more visible
    #                 )

    # Plot PB2003 plate boundaries in your region
    fig.plot(
        data="/home/staff/chao/SSEinv/PlateBoundary/pb2002_boundaries.gmt",
        region=inset_region, projection=inset_projection, pen="1p,black", style="f0.2c/3p", fill="black",)

    # Define plate label locations (lon, lat, label)
    plate_labels = [
        (-90.0, 8.0, "CO"), # Cocos
        (-85.0, 12.5, "CA"), # Caribbean
        (-83.0, 8.0, "PM"), # Panama     
        (-82.5, 5.5, "NZ"), # Nazca
        (-93.0, 14.0, "NA"), # North America
    ]

    # Add plate labels to the map
    for lon, lat, label in plate_labels:
        fig.text(x=lon, y=lat, text=label, font="10p,Helvetica-Bold,white", justify="LB",
                 region=inset_region, projection=inset_projection,)


    # Draw rectangle showing main map area
    rectangle_lon = [main_region[0], main_region[1], main_region[1], main_region[0], main_region[0]]
    rectangle_lat = [main_region[2], main_region[2], main_region[3], main_region[3], main_region[2]]

    fig.plot(x=rectangle_lon, y=rectangle_lat, pen="2p,blue")


fig.show()

figname = "nicoya_tectonic.pdf"
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/"
fig.savefig(figpath + figname)

print(f"Figure saved to: {figpath + figname}")